In [2]:
import numpy as np
import random
import matplotlib.pyplot as plt
from Bio import pairwise2
from Bio import SeqIO
import seaborn as sns

/home/Users/rdd4/conda/envs/ryan/lib/python3.12/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


# Mixed Sample Intrahost Variation Simulator

This is a super naive and simple simulator that can take a genome, mutate it, and then create 'reads' with settable parameters such as sequencing error, read length, etc. 

In [3]:
NUCLEOTIDES  = {'A', 'C', 'G', 'T'}

In [4]:
def write_random_genome(n:int, nucleotides=NUCLEOTIDES):
    """If want to simply write a completely random genome, instead of inputting a fasta file
    
    Writes a genome of length n (segmented genomes unavailable for now)

    Args:
        n (int): length of genome
        nucleotides (set, optional): set of characters to pull from for sequence. Defaults to NUCLEOTIDES.
        
    Returns:
        string: sequence of length n containing nucleotides
    """
    
    return ''.join(random.choices(list(nucleotides), k=n))

In [5]:
write_random_genome(10)

'GAAGACCAGA'

In [6]:
def input_genome(fasta:str):
    """Reads in a fasta file and returns the genome sequence (if segmented, will all be appended together)

    Args:
        fasta (str): location of fasta file
        
    Returns:
        string: sequence of genome within fasta file
    """
    seq = ""
    for record in SeqIO.parse(fasta, "fasta"):
        seq = seq + record.seq
    
    return str(seq)

In [7]:
input_genome('../test_data/covid.fasta')

'...'

In [8]:
def mutate_genome(genome:str, mutation_rate:float=0.01, generations:int=1, rates:list=[0.9, 0.05, 0.05]):
    """Returns evolved version of genome at given mutation rate over generations
    
    Args:
        genome(str): base genome
        mutation_rate(float): what percentage of genome will change after 1 generation (float between 0.0-1.0)
        generations (int): number of generations
        rates (list[tuple, tuple, tuple]): percentage of mutations that are mutations, insertions, and deletions
    """
    num_nt = len(genome)
    
    for _ in range(generations):
        for _ in range(int(mutation_rate*num_nt)):
            
            loc = np.random.randint(0, num_nt)
            mutation_type = np.random.choice(['MUTATION', 'INSERTION', 'DELETION'], p=rates)
            if mutation_type == 'MUTATION':
                genome = genome[:loc] + random.choice(list(NUCLEOTIDES)) + genome[loc+1:]
            elif mutation_type == 'INSERTION':
                insertion_length = int(1+np.random.exponential(0.5))
                genome = genome[:loc] + ''.join(random.choices(list(NUCLEOTIDES), k=insertion_length)) + genome[loc+1:]
            elif mutation_type == 'DELETION':
                deletion_length = int(1+np.random.exponential(0.5))
                genome = genome[:loc-int(deletion_length/2)] + genome[loc+1+int(deletion_length/2):]
    return genome

def get_strains(genome:str, n_strains:int=1, min_seq_identity:float=0.99, rates:list=[1.0, 0.0, 0.0]):
    """Will get multiple strains (of varying nt identity) to the seed genome (only SNP variation)
    
    genome: seed genome
    n_strains: number of strains that you want to generate
    min_seq_identity: how far the genomes can be from the seed (0.8 == 80% nt same) 
    
    """
    mutation_rates = np.random.rand(n_strains) * (1-min_seq_identity)
    genomes = [mutate_genome(genome, mutation_rate=mutation_rate, rates=rates) for mutation_rate in mutation_rates]
    return genomes

In [9]:
test = write_random_genome(n=300)
strains = get_strains(test, 5)
print(test)
for strain in strains:
    print(strain) if strain != test else print()

ACGGTCCCGTACATACGCCGAATGCGATGGGAGATGTTTCAGATAGAAGTCTCTCAATATGAGTCAACCGGATCATGCCTGGCTACTACGTTTGATAAACCAGTGCTCTACACGGTCGCAGTTTTTCGAACGGGCCCTCCCCTTGTTACGAGGTCCGACTGGCTACTAAACAGTGGTAGCCGATCGTGGCATACGCCTGTACGGAGGAATTGGGACATAGCTGACTTGGACCCCTCGGCCCAGCCAGGTACTACGGCGTAAAGATAAGGGCGCCTTGATGTTACGGATAGAGCAATGTTA



ACGGTCCCGTACATACGCCGAATGCGATGGGAGATGTTTCAGATAGAAGTCTCTCAATATGAGTCAACCGGATCATGCCTGGCTACTACGTTTGATAAACCAGTGCTCTACACGGTCGCAGTTTTTGGAACGGGCCCTCCCCTTGTTACGAGGTCCGACTGGCTACTAAACAGTGGTAGCCGATCGTGGCATACGCCTGTACGGAGGAATTGGGACATAGCTGACTTGGACCCCTCGGCCCAGCCAGGTACTACGGCGTAAAGATAAGGGCGCCTTGATGTTACGGATATAGCAATGTTA
ACGGTCCCGTACATACGCCGAATGCGATGGGAGATGTTTCAGATAGAAGTCTCTCAATATGTGTCAACCGGATCATGCCTGGCTACTACGTTTGATAAACCAGTGCTCTACACGGTCGCATTTTTTCGAACGGGCCCTCCCCTTGTTACGAGGTCCGACTGGCTACTAAACAGTGGTAGCCGATCGTGGCATACGCCTGTACGGAGGAATTGGGACATAGCTGACTTGGACCCCTCGGCCCAGCCAGGTACTACGGCGTAAAGATAAGGGCGCCTTGATGTTACGGATAGAGCAATGTTA


In [10]:
def sequence_genome(genome:str, read_length:int=150, depth:int=30, sequencing_error_rate=0.01, plot=False):
    """Returns read set after 'sequencing' genome, to certain average depth with sequencing error rate"""
    genome_length = len(genome)
    num_reads = int(genome_length / read_length * depth)
    seq_locs = np.random.randint(0, genome_length-read_length, size=num_reads)
    
    reads = []
    for i in seq_locs:
        read = genome[i:i+read_length]
        read_w_error = mutate_genome(read, sequencing_error_rate, rates=[1.0, 0.0, 0.0])
        reads.append(read_w_error)
        
    if plot:
        depths = [0]*genome_length
        for seq_loc in seq_locs:
            depths[seq_loc: seq_loc+read_length] = [x+1 for x in depths[seq_loc: seq_loc+read_length]]
        
        plt.bar(range(genome_length), depths)
        
    return reads

In [20]:
def metagenomic_sequencing(genome:str, n_strains:int=10, seq_similarity:float=0.99, read_length:int=150, min_depth:float=1.0, max_depth:float=100.0, sequencing_error_rate:int=0.01, strain_depths:list=[], output=False):
    """Performs 'metagenomic' sequencing on set of n_strains
    
    Args:
        genome (str): base genome to mutate into strains and sequence
        n_strains (int): the number of distinct lineages (for intrahost variation this is probably low, with high seq similarity)
        seq_similarity (float): how similar the strains will be to the base genome
        read_length (int): length of all reads (will be updated to have some variation)
        min_depth (float): minimum sequencing depth for any strain (>0)
        max_depth (float): depth of the original sequence (no other strains will be more abundant)
        sequencing_error_rate (flaot): error rate induced into reads
    """
    strains = get_strains(genome, n_strains, seq_similarity) + [genome]    
    
    if len(strain_depths) == n_strains:
        depths = strain_depths
    else:
        depths = np.random.lognormal(1, 1, size=n_strains)
        depths = np.append([max(int(min_depth), int(depth)/100 * max_depth) for depth in depths], max_depth)
    print(depths)
    
    reads = []
    for i, (strain, depth) in enumerate(zip(strains, depths)):
        reads += sequence_genome(strain, read_length=read_length, depth=depth, sequencing_error_rate=sequencing_error_rate)
        if output:
            print(f'Sequenced strain {i} at depth {depth}')
    return reads, strains, depths

In [23]:
reads, strains, depths = metagenomic_sequencing(input_genome('../test_data/covid.fasta'), seq_similarity=0.9999, n_strains=2, output=True, max_depth=500, sequencing_error_rate=0.000)

[ 10.  40. 500.]
Sequenced strain 0 at depth 10.0
Sequenced strain 1 at depth 40.0
Sequenced strain 2 at depth 500.0


In [13]:
len(reads)

109643

In [14]:
strains

['...']

In [17]:
def write_fasta(sequences, filename, ids=[]):
    """Writes a list of DNA sequences to a FASTA file.

    Args:
        sequences: A list of DNA sequence strings.
        ids: A list of sequence identifiers (one for each sequence).
        filename: The name of the output FASTA file.
    """
    if len(ids)==0:
        ids = range(len(sequences))
        
    with open(filename, 'w') as f:
        for i, seq in enumerate(sequences):
            f.write(">" + str(ids[i]) + "\n")
            f.write(seq + "\n")
            
def write_fastq(sequences, filename, ids=[]):
    """Writes a list of DNA sequences to a FASTQ file with best quality scores.

    Args:
        sequences: A list of DNA sequence strings.
        ids: A list of sequence identifiers (one for each sequence).
        filename: The name of the output FASTQ file.
    """
    if len(ids) == 0:
        ids = range(len(sequences))
    
    with open(filename, 'w') as f:
        for i, seq in enumerate(sequences):
            # Write the FASTQ entry
            f.write("@" + str(ids[i]) + "\n")  # Sequence identifier
            f.write(seq + "\n")                # Sequence
            f.write("+\n")                     # Separator
            f.write('I' * len(seq) + "\n")     # Quality score (all 'I' for best quality)


In [ ]:
write_fasta(reads, '../test_data/test3.fasta')

In [24]:
write_fastq(reads, '../test_data/test3.fastq')

## Testing kmer distributions in sequencing data

In [151]:
def check_seq_identity(genomes):
    """Tests the sequence identity is correct for a set of strains

    Args:
        genomes (list[str]): list of genomes

    Returns:
        np.matrix: 2d matrix of sequence identities
    """
    
    identities = np.zeros(shape=(len(genomes), len(genomes)))
    
    for i, g1 in enumerate(genomes):
        for j, g2 in enumerate(genomes[i:]):
            alignments = pairwise2.align.globalxx(g1, g2)
            num_matches = alignments[0][2]
            seq_identity = np.round(num_matches / max(len(g1), len(g2)) * 100, 1)

            identities[i,j+i] = seq_identity
            identities[j+i,i] = seq_identity
        
    return identities

In [152]:
import pickle

with open('../kmer.pkl', 'rb') as file:
    kmer_index = pickle.load(file)


In [153]:
kmer_index

defaultdict(list,
            {1142000672: [(0, 0, 'A')],
             269765458: [(0, 1, 'T')],
             924076882: [(0, 2, 'T')],
             1142000673: [(0, 3, 'A')],
             1142000674: [(0, 4, 'A')],
             1142000675: [(0, 5, 'A')],
             1141197653: [(0, 6, 'G')],
             1141812053: [(0, 7, 'G')],
             1141932885: [(0, 8, 'T')],
             1141984341: [(0, 9, 'T')],
             1141996629: [(0, 10, 'T')],
             1142000676: [(0, 11, 'A')],
             1142000398: [(0, 12, 'T')],
             1142000677: [(0, 13, 'A')],
             1142000671: [(0, 14, 'C')],
             927526837: [(1, 0, 'T')],
             3091787701: [(1, 1, 'T')],
             3812487920: [(1, 2, 'A')],
             3812487921: [(1, 3, 'A')],
             3812487922: [(1, 4, 'A')],
             3809669048: [(1, 5, 'G')],
             3811831736: [(1, 6, 'G')],
             3812253624: [(1, 7, 'T')],
             3812431800: [(1, 8, 'T')],
             3812474

In [53]:
test = [[0]*4 for _ in range(30000)]
test[0][0] += 1
test

[[1, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0,

In [ ]:
output = np.zeros((30000, 4), dtype=int)
# print(output)
k=20
buckets = [1491761659837, 340894762419, 1491761659838, 1491761658536]
kmer_bin = 270847648280

for bucket in buckets:
    if len(kmer_index[bucket])==1: ##only unique hits
        genome_pos, nuc_x = kmer_index[bucket][0]
        mask = 0b11 << (k-nuc_x-1) * 2
        extracted_bits = (kmer_bin & mask) >> (k-nuc_x-1) * 2
        print(bin(kmer_bin)[2:].zfill(k*2), bin(mask)[2:].zfill(k*2), genome_pos, nuc_x, bin(extracted_bits))
        output[genome_pos + nuc_x, int(extracted_bits)] += 1
        
output[:20]

0011111100001111110001110010001000011000 1100000000000000000000000000000000000000 0 0 0b0
0011111100001111110001110010001000011000 0011000000000000000000000000000000000000 0 1 0b11
0011111100001111110001110010001000011000 0000001100000000000000000000000000000000 0 3 0b11
0011111100001111110001110010001000011000 0000000000000000000000000000000011000000 0 16 0b0
0011111100001111110001110010001000011000 0000001100000000000000000000000000000000 0 3 0b11


array([[1, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 0, 0],
       [0, 0, 0, 2],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]])

In [ ]:
output

array([[1, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       ...,
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]])

In [136]:
def invert_kbit_integer(n, k):
    # Create a bitmask with k bits set to 1 (all 1s of length k)
    bitmask = (1 << k*2) - 1
    return ~n & bitmask

In [139]:
n = 174
print(bin(n))
bin(invert_kbit_integer(n, k=8))

0b10101110


'0b1111111101010001'